In [4]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import asyncio
from pathlib import Path
from llama_index.core import VectorStoreIndex, SimpleDirectofrom llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.huggingface_api import HuggingFaceInffrom llama_index.embeddings.huggingface import HuggingFaceInferenceAPIfrom retrieval.rag_helper import RAGHelperEmbeddingerenceAPI
ryReader



# Add the src directory to the Python path
sys.path.append(str(Path().resolve().parent / "src"))
import nest_asyncio

nest_asyncio.apply()

with open((Path().resolve().parent / "hf_token.json").as_posix()) as f:
    HF_TOKEN: str = json.load(f)["HUGGING_FACE_TOKEN"]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LLM Call

In [ ]:
llm: HuggingFaceInferenceAPI = HuggingFaceInferenceAPI(
    model_name="deepseek-ai/DeepSeek-V3-0324",
    token=HF_TOKEN,
    provider="auto",  # this will use the best provider available
    temperature=0.1,  #
)

print(llm.complete(prompt="What is the capital of France?").text)

Embedding Model

In [6]:

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


embed_model = HuggingFaceInferenceAPIEmbedding(model_name=MODEL_NAME, token=HF_TOKEN)
# --- 2. Get Embeddings ---
sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "Exploring the solar system is a great adventure.",
]

# Use the proper public method for getting # embeddings
embe# ddings = []
for sentence i# n sentences:
    embedding = embed_model.get_text_embedd# ing(sentence)
    embeddings.appe# nd(embedding)

print(f"Generated {len(embedding# s)} embeddings")
print(f"Embedding dimension: {len(embeddings[0]) if embeddings else 'N/A'}")

C:\Users\Valentin\AppData\Local\Temp\ipykernel_14180\2100356627.py:4: DeprecationWarning: Call to deprecated class HuggingFaceInferenceAPIEmbedding. (Deprecated in favor of `HuggingFaceInferenceAPIEmbedding` from `llama-index-embeddings-huggingface-api` which should be used instead.)
  embed_model = HuggingFaceInferenceAPIEmbedding(model_name=MODEL_NAME, token=HF_TOKEN)


RAG Part

In [63]:
rag_helper = RAGHelper(
    llm=llm,
    text_qa_template="Y oau a re  a ",
    embed_model=embed_model,
    hf_token=HF_TOKEN,
)
document_path: str = (Path().resolve().parent /datacs").as_posix()

doc_lst = rag_helper.load_documents(directory_path=document_path)

rag_helper.build_index(documents=doc_lst)

query_engine = rag_helper.get_query_engine()

In [64]:
rag_helper.build_index(directory_path=document_path)

ClientResponseError: 402, message='Payment Required', url='https://router.huggingface.co/hf-inference/models/sentence-transformers/all-MiniLM-L6-v2/pipeline/feature-extraction'

In [24]:
import pymupdf4llm
directory_path: str = (Path().resolve().parent / "data").as_posix()
docs = []

llama_reader = pymupdf4llm.LlamaMarkdownReader()
# Load PDF documents, convert them to markdown and write images into the same directory
for file in [
    f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))
]:
    docs.extend(
        llama_reader.load_data(
            file_path=os.path.join(directory_path, file),
            image_path=directory_path,
            write_images=True,
            image_format="png",
        )
    )

# Add generated images to the documents list
docs.extend(SimpleDirectoryReader(directory_path, required_exts=[".png"]).load_data())

In [16]:
rag.query("Hello")p

In [25]:
fraom llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import ImageReader, PDFReader

file_extractor = {
    ".png": ImageReader(keep_image=True),
    ".jpeg": ImageReader(keep_image=True),
    ".jpg": ImageReader(keep_image=True),
    ".pdf": PDFReader(return_full_document=True),  # PDFReader should have image extraction enabled
}


document_path: str = (Path().resolve().parent / "data").as_posix()

docs = SimpleDirectoryReader(document_path, file_extractor=file_extractor).load_data()

In [40]:
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.core.schema import BaseNode
from transformers import AutoTokenizer

import tiktoken

splitter: MarkdownNodeParser = MarkdownNodeParser(
    include_metadata=True, include_prev_next_rel=True
)

nodes: list[BaseNode] = splitter.get_nodes_from_documents(docs)

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3-0324")

token_aware_splitter = SentenceSplitter(
    chunk_size=500,
    chunk_overlap=100,
    tokenizer=tokenizer.encode,
)

final_nodes = token_aware_splitter.get_nodes_from_documents(nodes)

In [48]:
final_nodes

[TextNode(id_='6a1d47d5-ef2e-409a-8010-6d12711978d0', embedding=None, metadata={'file_name': 'cahier-charges-techs-environnementales.pdf', 'file_path': 'C:\\Users\\Valentin\\Projects\\DS_Project_Test\\data\\cahier-charges-techs-environnementales.pdf', 'file_type': 'application/pdf', 'file_size': 7356447, 'creation_date': '2025-10-11', 'last_modified_date': '2025-10-11', 'header_path': '/'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='c3719f52-ab8f-4b19-a26e-5c33c5855789', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_name': 'cahier-charges-techs-environnementales.pdf', 'file_path': 'C:\\Users\\Valentin\\Projects\\DS_Project_Test\\data\\cahier-charges-techs-environnementales.pdf', 'file_type': 'a

In [17]:
from llama_index.core.schema import ImageDocument

[d for d in docs if isinstance(d, ImageDocument)]

[]

In [ ]:
doc_lst = rag_helper.load_documents(directory_path=document_path)

rag_helper.build_index(documents=doc_lst)

query_engine = rag_helper.get_query_engine()

In [40]:
query_engine.query(
    "De quelle ville est issu ce cahier des charges techniques et environementales ?"
)

Response(response='Ce cahier des charges techniques et environnementales est issu de la ville de Lyon.', source_nodes=[NodeWithScore(node=TextNode(id_='c2ce5b19-1aeb-4abf-90f8-3d8f337e9a85', embedding=None, metadata={'page_label': '2', 'file_name': 'cahier-charges-techs-environnementales.pdf', 'file_path': 'C:\\Users\\Valentin\\Projects\\DS_Project_Test\\docs\\cahier-charges-techs-environnementales.pdf', 'file_type': 'application/pdf', 'file_size': 7356447, 'creation_date': '2025-10-11', 'last_modified_date': '2025-10-11'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='e060a42b-81d1-4cb0-b841-cba3ed1d807b', node_type='4', metadata={'page_label': '2', 'file_name': 'cahier-charges-techs-environnementales.pd